In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
dataset_path = r"C:\Student-Exam-Score-Predictor\dataset\student_data_cleaned.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (6607, 20)


In [4]:
X = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (6607, 19)
Target shape: (6607,)


In [5]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

Categorical features:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


C:\Users\Chotu\AppData\Local\Temp\ipykernel_30376\1998881318.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [6]:
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 5285
Testing samples: 1322


In [8]:
linear_regression_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

print("Linear Regression model created!")

Linear Regression model created!


In [9]:
linear_regression_model.fit(X_train, y_train)

print("Linear Regression training completed!")

Linear Regression training completed!


In [10]:
lr_predictions = linear_regression_model.predict(X_test)

print("Linear Regression predictions generated!")
print(lr_predictions[:10])

Linear Regression predictions generated!
[64.52628302 65.2651138  71.53126265 64.27748146 66.5216532  66.61246339
 72.47094891 66.49826901 69.98604472 70.13327106]


In [11]:
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_rmse = np.sqrt(lr_mse)
lr_r2 = r2_score(y_test, lr_predictions)

print("Linear Regression Results")
print("-------------------------")
print("MSE :", lr_mse)
print("RMSE:", lr_rmse)
print("R²  :", lr_r2)

Linear Regression Results
-------------------------
MSE : 3.2560199870428943
RMSE: 1.804444509272284
R²  : 0.7696495724907312


In [12]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest Regression model created!")

Random Forest Regression model created!


In [13]:
random_forest_model.fit(X_train, y_train)

print("Random Forest Regression training completed!")

Random Forest Regression training completed!


In [14]:
rf_predictions = random_forest_model.predict(X_test)

print("Random Forest predictions generated!")
print(rf_predictions[:10])

Random Forest predictions generated!
[64.685 66.35  70.26  65.775 66.26  66.625 71.02  66.855 69.655 70.075]


In [15]:
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Regression Results")
print("--------------------------------")
print("MSE :", rf_mse)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

Random Forest Regression Results
--------------------------------
MSE : 4.660667397881997
RMSE: 2.1588578920072523
R²  : 0.6702763705834445


In [16]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest Regression"
    ],
    "MSE": [
        lr_mse,
        rf_mse
    ],
    "RMSE": [
        lr_rmse,
        rf_rmse
    ],
    "R2": [
        lr_r2,
        rf_r2
    ]
})

results

,Model,MSE,RMSE,R2
0,Linear Regression,3.256020,1.804445,0.769650
1,Random Forest Regression,4.660667,2.158858,0.670276


In [17]:
if rf_r2 > lr_r2:
    best_model = random_forest_model
    best_model_name = "Random Forest Regression"
else:
    best_model = linear_regression_model
    best_model_name = "Linear Regression"

print("Best Model:", best_model_name)

Best Model: Linear Regression


In [18]:
model_path = r"C:\Student-Exam-Score-Predictor\models\best_model.pkl"

joblib.dump(best_model, model_path)

print("Best model saved successfully!")
print(model_path)

Best model saved successfully!
C:\Student-Exam-Score-Predictor\models\best_model.pkl
